# NPZ Viewer — inspect any feature batch and its labels

Set the three options in the **config cell** below, then **run it** (Shift+Enter).
Change `BATCH_NUM` and re-run to look at a different file.

- `SOURCE`  : `"author"` (reference features) or `"generated"` (your own output)
- `SPLIT`   : `"train"` or `"test"`
- `BATCH_NUM`: which batch file, e.g. `1` → `batch_001.npz`

## Config + load + summary

In [41]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

# ----------------- CHOOSE WHAT TO VIEW -----------------
SOURCE    = "generated"      # "author"  or  "generated"
SPLIT     = "train"        # "train"   or  "test"
BATCH_NUM = 1             # 1 -> batch_001.npz
# -------------------------------------------------------

base = "../Dataset/features" if SOURCE == "author" else "../Dataset/features_generated"
path = f"{base}/{SPLIT}/batch_{BATCH_NUM:03d}.npz"

if not os.path.exists(path):
    print(f"NOT FOUND: {path}\n")
    avail = sorted(glob.glob(f"{base}/{SPLIT}/batch_*.npz"))
    print(f"available in {base}/{SPLIT}/ :", len(avail), "files")
    if avail:
        print("  first:", os.path.basename(avail[0]), " last:", os.path.basename(avail[-1]))
else:
    f = np.load(path)
    data, labels = f["data"], f["labels"]
    print("file  :", path)
    print("keys  :", f.files)
    print("data  :", data.shape, data.dtype)
    print("labels:", labels.shape, labels.dtype)
    print("values: min=%.3f max=%.3f mean=%.4f  (>=0? %s)" % (
        data.min(), data.max(), data.mean(), bool((data >= 0).all())))

file  : ../Dataset/features_generated/train/batch_001.npz
keys  : ['data', 'labels']
data  : (10, 100, 20, 4096) float16
labels: (10, 2) float16
values: min=0.000 max=6.586 mean=0.0059  (>=0? True)


## Labels: per-video + batch composition

In [42]:
BATCH_NUM = 1  
f = np.load(path)
labels = f["labels"]

n_acc = int(sum(1 for r in labels if r[1] == 1))
print(f"composition: {n_acc} accident  +  {len(labels)-n_acc} non-accident\n")
print(" video | label  | class")
print(" ------+--------+------------")
for i, r in enumerate(labels):
    kind = "ACCIDENT" if r[1] == 1 else "no-accident"
    print(f"   {i:2d}  | {list(r)} | {kind}")

order = ["A" if r[1] == 1 else "n" for r in labels]
print("\norder:", order)

composition: 5 accident  +  5 non-accident

 video | label  | class
 ------+--------+------------
    0  | [1.0, 0.0] | no-accident
    1  | [1.0, 0.0] | no-accident
    2  | [1.0, 0.0] | no-accident
    3  | [0.0, 1.0] | ACCIDENT
    4  | [0.0, 1.0] | ACCIDENT
    5  | [0.0, 1.0] | ACCIDENT
    6  | [1.0, 0.0] | no-accident
    7  | [1.0, 0.0] | no-accident
    8  | [0.0, 1.0] | ACCIDENT
    9  | [0.0, 1.0] | ACCIDENT

order: ['n', 'n', 'n', 'A', 'A', 'A', 'n', 'n', 'A', 'A']


In [39]:
FRAME = 50
f = np.load(path)
print(f"data[{VIDEO}, {FRAME}, 0, :20] =")
print(np.round(f['data'][VIDEO, FRAME, 0, :20], 3))

data[0, 50, 0, :20] =
[0.204 0.014 0.    0.179 0.512 0.    0.    0.426 0.    0.    0.    0.
 0.    0.    0.    0.45  0.    1.369 0.661 0.02 ]


## Compare order across many batches
Scans a range of batch files and prints each one's accident/non order, so you can compare them at a glance. Set the range and re-run.

In [44]:
# Compare accident/non order across a RANGE of batches
SCAN_SOURCE = "generated"      # "author" or "generated"
SCAN_SPLIT  = "train"        # "train" or "test"
SCAN_FROM, SCAN_TO = 1,10  # inclusive batch range to compare

scan_base = "../Dataset/features" if SCAN_SOURCE == "author" else "../Dataset/features_generated"
print(f"{SCAN_SOURCE}/{SCAN_SPLIT}  batches {SCAN_FROM}..{SCAN_TO}")
print(" batch | acc | order")
print(" ------+-----+------------------------------")
seen = {}
for b in range(SCAN_FROM, SCAN_TO + 1):
    p = f"{scan_base}/{SCAN_SPLIT}/batch_{b:03d}.npz"
    if not os.path.exists(p):
        print(f"  {b:3d}  |  -  | (missing)")
        continue
    lab = np.load(p)["labels"]
    order = "".join("A" if r[1] == 1 else "n" for r in lab)
    print(f"  {b:3d}  |  {order.count('A')}  | {order}")
    seen[order] = seen.get(order, 0) + 1

print("distinct orders found:")
for o, c in seen.items():
    print(f"   {o}  x{c}")

generated/train  batches 1..10
 batch | acc | order
 ------+-----+------------------------------
    1  |  5  | nnnAAAnnAA
    2  |  -  | (missing)
    3  |  -  | (missing)
    4  |  -  | (missing)
    5  |  -  | (missing)
    6  |  -  | (missing)
    7  |  -  | (missing)
    8  |  -  | (missing)
    9  |  -  | (missing)
   10  |  -  | (missing)
distinct orders found:
   nnnAAAnnAA  x1


**Tips:** change `SOURCE`/`SPLIT`/`BATCH_NUM` in the config cell and re-run it, then re-run the cells below. `author` has train (128) + test (46) batches; `generated` has whatever you built in Step 6.